# Solution: run the shuffle on a question the session did not answer

The session tested Retail-Plus against Retail-Core and found the gap real. Marketing asks the same
question about their own cut:

> "On Tuesday you showed me web down 22 percent and app flat. You have since cleaned the data. Is
> that web fall still there, and is it real?"

Every blank is filled and the notebook is executed, so the numbers below are the ones the file
actually produces with this seed.

In [1]:
import pathlib
import random
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

ORDERS = kit.load_csv("C2_W01_D04_orders_STUDENT.csv")
print(f"{len(ORDERS)} cleaned orders")

186 cleaned orders


## 1. Which of the three habits does this question need?

Light the branch, and be ready to say why the other two do not fit.

In [2]:
kit.tree(
    {"label": "is the web fall real?",
     "branches": [
         ("could chance do it", {"label": "a chance reference"}),
         ("is the base big enough", {"label": "sample size"}),
         ("did web cause it", {"label": "a fair comparison"}),
     ]},
    taken=["could chance do it"], title="Pick the habit before writing any code")

## 2. Measure the real gap first

Never shuffle before you know what you are comparing against.

In [3]:
def fall(rows):
    q1 = [r for r in rows if r["quarter"] == "Q1"]
    q2 = [r for r in rows if r["quarter"] == "Q2"]
    a = len(q1) / len({r["customer_id"] for r in q1})
    b = len(q2) / len({r["customer_id"] for r in q2})
    return 100 * (b / a - 1)


web = [r for r in ORDERS if r["channel"] == "web"]
app = [r for r in ORDERS if r["channel"] == "app"]
observed = fall(app) - fall(web)
print(f"web fell {fall(web):.1f} percent, app fell {fall(app):.1f} percent")
print(f"the gap: {observed:.1f} points")

web fell 3.4 percent, app fell 0.0 percent
the gap: -3.4 points


In [4]:
kit.check("both channels have orders in both quarters", len(web) > 20 and len(app) > 20,
          f"web {len(web)}, app {len(app)}")

## 3. Shuffle the labels, keeping the group sizes

The unit you shuffle is the **customer**, not the order, because a customer's orders belong
together. Getting this wrong is the most common mistake in the exercise.

In [5]:
by_customer = {}
for r in web + app:
    by_customer.setdefault(r["customer_id"], []).append(r)
members = sorted(by_customer)
n_web = len({r["customer_id"] for r in web})


def shuffled_gap(rng):
    m = members[:]
    rng.shuffle(m)
    a = [r for c in m[:n_web] for r in by_customer[c]]
    b = [r for c in m[n_web:] for r in by_customer[c]]
    return fall(b) - fall(a)


rng = random.Random(20260101)
ten = [shuffled_gap(rng) for _ in range(10)]
print([round(g, 1) for g in ten])

[-4.1, 11.1, -9.3, -6.4, 15.0, 4.6, -7.0, -7.3, -13.0, -19.6]


## 4. Five thousand of them, and the sentence

In [6]:
rng = random.Random(20260101)
N = 5000
extreme = sum(1 for _ in range(N) if abs(shuffled_gap(rng)) >= abs(observed))
print(f"{extreme} of {N} at least as extreme")
# Write the p-value the way it should be written, given how many shuffles you ran:
p_sentence = (f"p = {extreme / N:.4f}" if extreme
              else f"p < {1 / N:.4f}, which is the resolution of 5,000 shuffles")
print(p_sentence)

3737 of 5000 at least as extreme
p = 0.7474


In [7]:
kit.check("you counted the extreme shuffles", isinstance(extreme, int), f"{extreme}")
kit.check("you shuffled customers rather than orders", len(members) < len(web + app),
          f"{len(members)} customers behind {len(web + app)} orders")
kit.check("your sentence reports a bound when nothing was extreme, and a share otherwise",
          ("<" in p_sentence) if extreme == 0 else ("<" not in p_sentence),
          f"{p_sentence!r} with {extreme} extreme of {N}")

## 5. Rank four sentences by how defensible they are

Put them in order, most defensible at the top, and mark the line below which you would not write
any of them in a note to a CEO.

In [8]:
kit.decision_ladder(
    ["chance produced a gap this large in none of 5,000 shuffles, so p is below 0.0002",
     "the gap is unlikely to be chance",
     "there is a 0.02 percent chance the finding is wrong",
     "the test proves the web channel caused the fall"],
    cut_at=2, title="Four p-value sentences, ranked, with the line below which none is defensible")

## 5b. Draw the move you just made

Four boxes: what you measured, what you assumed, what you counted, what you concluded.

In [9]:
kit.flow(["measure the real gap",
          "assume the labels mean nothing",
          "count the chance worlds that beat it",
          "report the share, at your resolution"], lit=1,
         title="The permutation test in four steps")

## 6. The five letters to post

**Q1.** `p = 0.03`. Which reading is right?
`a` a 3 percent chance the finding is wrong · `b` a 3 percent chance chance caused it ·
`c` 3 percent of chance-only worlds are at least this extreme · `d` the effect is 3 percent in size

**Q2.** You ran 5,000 shuffles and none was as extreme. What do you write?
`a` p = 0 · `b` p < 0.0002 · `c` p = 0.0002 · `d` the result is certain

**Q3.** You shuffled orders rather than customers. What did that do?
`a` nothing, an order is the unit of analysis here ·
`b` it split some customers across both labels, so the null is wrong ·
`c` it made the p-value larger and therefore more conservative ·
`d` it only matters if customers have different numbers of orders

**Q4.** A tiny p-value on a two percent difference. What do you tell Meera?
`a` it is significant, so act on it · `b` it is not significant because the effect is small ·
`c` it is real and probably too small to be worth acting on · `d` rerun with more shuffles

**Q5.** Your p-value comes out large. What do you tell marketing?
`a` the web fall was never real and Tuesday was a mistake ·
`b` the gap on clean data is the size chance produces routinely ·
`c` run more shuffles, because 5,000 was not enough to resolve it ·
`d` the test failed, so the question cannot be answered from this file

In [10]:
my_answers = "cbbcb"
kit.check("five letters posted", len(my_answers) == 5 and my_answers.isalpha(),
          f"got {my_answers!r}")
kit.check_summary()

## 7. Two sentences for the note

One that reports what you found on clean data. One that says what happened to Tuesday's number and
why.

> "On the reconciled data web orders per customer are up 3.4 percent and app is flat, a gap of 3.4
> points. Shuffling the same customers between the two labels produced a gap at least that large in
> about three quarters of 5,000 chance-only worlds, so there is no channel difference to explain."

> "Tuesday's 22 percent web fall was computed before the reconciliation, and most of it was the
> duplicated rows, which sat in Retail-Plus and therefore in web. The segment finding survived the
> clean pass and the channel finding did not."

## Why each letter

| | Key | Why the others fail |
|---|---|---|
| **Q1** | `c` | The p-value is a property of the chance-only worlds, not of your finding. `a` and `b` both turn it into a probability about the conclusion, which reverses what it measures, and `a` is the one that gets said aloud in meetings. |
| **Q2** | `b` | Five thousand shuffles resolve to one in five thousand. `a` claims a certainty the method cannot produce, and `c` reports a number you did not observe: none of the shuffles reached it, so the answer is an upper bound rather than a point. |
| **Q3** | `b` | A customer's orders belong together. Shuffling orders can put the same customer's Q1 order under one label and their Q2 order under the other, which makes the null world one that could not exist. The p-value it produces is not wrong by a little; it answers a different question. |
| **Q4** | `c` | Real and worth acting on are separate calls. A tiny p-value on a two percent difference is a precise measurement of something small. `b` is the mirror-image error: significance says nothing about size in either direction. |
| **Q5** | `b` | A large p-value says the gap is the size chance produces routinely, which is all it says. `a` overclaims in the other direction and calls a colleague's work a mistake when it was a correct computation on the data available that day. `c` is the instinct to keep running until the answer changes, and `d` treats a clear result as a failure because it is not the expected one. |

## The thing worth taking away

Tuesday's channel cut was not wrong. It was computed on the export as it arrived, and most of the
web fall was the duplicated rows, which sat in Retail-Plus and therefore in web.

The segment finding survived the clean pass at 33 percent. The channel finding did not survive at
all. **A cut that holds on one day's data can vanish on the next**, and the only way to know is to
re-run it rather than to reuse the slide.